# 0. Preprocessing

## Aim
- Parse musical note events from raw MIDI files.
- Convert each track into two learning representations:
  - Token/event sequences for sequence models.
  - Piano-roll matrices for time-step models.
- Create deterministic train/validation/test splits.

## Why this design
- Raw MIDI is the correct primary source for generation-ready note events.
- Chunked saving avoids high memory usage on large datasets.
- Progress bars to prevent panic :).

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import random
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import mido
import numpy as np
from tqdm.auto import tqdm

try:
    import h5py
    H5PY_AVAILABLE = True
    print("Ase bhai")
except Exception:
    h5py = None
    H5PY_AVAILABLE = False

Ase bhai


In [32]:
@dataclass
class PreprocessConfig:
    project_root: Path = Path("..").resolve()
    raw_midi_dir: Path = project_root / "data" / "raw_midi"
    lmd_h5_dir: Path = project_root / "data" / "dataset"
    output_dir: Path = project_root / "data" / "processed"

    use_raw_midi: bool = True
    use_lmd_h5: bool = False

    max_files: Optional[int] = None
    random_seed: int = 42

    fs: int = 16
    max_roll_steps: int = 2048
    min_notes_per_track: int = 8

    duration_bins: Tuple[float, ...] = (0.1, 0.25, 0.5, 1.0, 2.0, 4.0)
    velocity_bins: Tuple[int, ...] = (16, 32, 48, 64, 80, 96, 112, 127)

    test_size: float = 0.10
    val_size_from_train: float = 0.10

    chunk_size: int = 256
    checkpoint_every: int = 500
    merge_final_npz: bool = False
    show_progress: bool = True
    save_compressed: bool = True


config = PreprocessConfig()
config.output_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in asdict(config).items()}, indent=2))

{
  "project_root": "G:\\Unsupervised-Neural-Network-for-Multi-Genre-Music-Generation-CSE425-",
  "raw_midi_dir": "G:\\Unsupervised-Neural-Network-for-Multi-Genre-Music-Generation-CSE425-\\data\\raw_midi",
  "lmd_h5_dir": "G:\\Unsupervised-Neural-Network-for-Multi-Genre-Music-Generation-CSE425-\\data\\dataset",
  "output_dir": "G:\\Unsupervised-Neural-Network-for-Multi-Genre-Music-Generation-CSE425-\\data\\processed",
  "use_raw_midi": true,
  "use_lmd_h5": false,
  "max_files": null,
  "random_seed": 42,
  "fs": 16,
  "max_roll_steps": 2048,
  "min_notes_per_track": 8,
  "duration_bins": [
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0
  ],
  "velocity_bins": [
    16,
    32,
    48,
    64,
    80,
    96,
    112,
    127
  ],
  "test_size": 0.1,
  "val_size_from_train": 0.1,
  "chunk_size": 256,
  "checkpoint_every": 500,
  "merge_final_npz": false,
  "show_progress": true,
  "save_compressed": true
}


In [33]:
quick_validate_max_files: Optional[int] = None
if quick_validate_max_files is not None:
    config.max_files = quick_validate_max_files
    print(f"Quick validation enabled: max_files={config.max_files}")

In [ ]:
MIDI_EXTENSIONS = {".mid", ".midi", ".MID", ".MIDI"}


def discover_files(root: Path, allowed_suffixes: Optional[Sequence[str]] = None, limit: Optional[int] = None) -> List[Path]:
    if not root.exists():
        return []
    if allowed_suffixes is None:
        files = [p for p in root.rglob("*") if p.is_file()]
    else:
        suffixes = set(allowed_suffixes)
        files = [p for p in root.rglob("*") if p.is_file() and p.suffix in suffixes]
    files.sort()
    return files[:limit] if limit is not None else files


def find_input_files(cfg: PreprocessConfig) -> Dict[str, List[Path]]:
    inputs: Dict[str, List[Path]] = {"raw_midi": [], "lmd_h5": []}
    if cfg.use_raw_midi:
        inputs["raw_midi"] = discover_files(cfg.raw_midi_dir, list(MIDI_EXTENSIONS), cfg.max_files)
    if cfg.use_lmd_h5:
        inputs["lmd_h5"] = discover_files(cfg.lmd_h5_dir, [".h5", ".hdf5"], cfg.max_files)
    return inputs


def inspect_first_h5_schema(input_index: Dict[str, List[Path]], max_items: int = 20) -> None:
    if not H5PY_AVAILABLE:
        print("h5py not available.")
        return
    h5_files = input_index.get("lmd_h5", [])
    if not h5_files:
        print("No H5 files discovered.")
        return
    target = h5_files[0]
    print("Inspecting:", target)
    with h5py.File(target, "r") as f:
        items = []
        f.visititems(lambda name, obj: items.append((name, type(obj).__name__, getattr(obj, "shape", None))))
        for i, (name, typ, shape) in enumerate(items[:max_items]):
            print(f"{i+1:02d}. {name} | {typ} | shape={shape}")


def parse_raw_midi_file(path: Path) -> Optional[List[Tuple[int, float, float, int]]]:
    try:
        midi = mido.MidiFile(str(path))
    except Exception:
        return None

    current_time = 0.0
    active: Dict[Tuple[int, int], List[Tuple[float, int]]] = defaultdict(list)
    notes: List[Tuple[int, float, float, int]] = []

    try:
        for msg in midi:
            current_time += float(msg.time)
            if msg.type == "note_on" and getattr(msg, "velocity", 0) > 0:
                key = (int(msg.note), int(getattr(msg, "channel", 0)))
                active[key].append((current_time, int(msg.velocity)))
            elif msg.type == "note_off" or (msg.type == "note_on" and getattr(msg, "velocity", 0) == 0):
                key = (int(msg.note), int(getattr(msg, "channel", 0)))
                if active[key]:
                    start, velocity = active[key].pop(0)
                    if current_time > start:
                        notes.append((int(msg.note), float(start), float(current_time), int(velocity)))
    except Exception:
        return None

    notes.sort(key=lambda x: (x[1], x[0]))
    return notes if notes else None


def parse_lmd_h5_file(path: Path) -> Optional[List[Tuple[int, float, float, int]]]:
    return None


def quantize_duration(duration: float, bins: Sequence[float]) -> int:
    for i, b in enumerate(bins):
        if duration <= b:
            return i
    return len(bins)


def quantize_velocity(velocity: int, bins: Sequence[int]) -> int:
    for i, b in enumerate(bins):
        if velocity <= b:
            return i
    return len(bins)


def notes_to_tokens(
    notes: Sequence[Tuple[int, float, float, int]],
    duration_bins: Sequence[float],
    velocity_bins: Sequence[int],
) -> List[int]:
    tokens: List[int] = []
    for pitch, start, end, velocity in notes:
        duration = max(0.01, end - start)
        d_bin = quantize_duration(duration, duration_bins)
        v_bin = quantize_velocity(velocity, velocity_bins)
        token = int(pitch) + 128 * d_bin + 128 * (len(duration_bins) + 1) * v_bin
        tokens.append(token)
    return tokens


def notes_to_pianoroll(
    notes: Sequence[Tuple[int, float, float, int]],
    fs: int,
    max_steps: int,
) -> np.ndarray:
    if not notes:
        return np.zeros((max_steps, 128), dtype=np.float32)
    max_end = max(end for _, _, end, _ in notes)
    total_steps = min(max_steps, max(1, int(math.ceil(max_end * fs))))
    roll = np.zeros((total_steps, 128), dtype=np.float32)
    for pitch, start, end, velocity in notes:
        start_idx = max(0, int(math.floor(start * fs)))
        end_idx = min(total_steps, max(start_idx + 1, int(math.ceil(end * fs))))
        if start_idx >= total_steps:
            continue
        value = float(velocity) / 127.0
        roll[start_idx:end_idx, pitch] = np.maximum(roll[start_idx:end_idx, pitch], value)
    if total_steps < max_steps:
        padded = np.zeros((max_steps, 128), dtype=np.float32)
        padded[:total_steps] = roll
        return padded
    return roll


def split_name_for_example(path_str: str, cfg: PreprocessConfig) -> str:
    stable_key = f"{path_str}|{cfg.random_seed}".encode("utf-8")
    stable_int = int.from_bytes(hashlib.sha1(stable_key).digest()[:8], byteorder="big", signed=False)
    rnd = random.Random(stable_int)
    r = rnd.random()
    if r < cfg.test_size:
        return "test"
    val_threshold = cfg.test_size + (1.0 - cfg.test_size) * cfg.val_size_from_train
    if r < val_threshold:
        return "val"
    return "train"


def flush_chunk(
    cfg: PreprocessConfig,
    split_name: str,
    chunk_id: int,
    rows: List[dict],
    manifests: Dict[str, List[str]],
) -> None:
    if not rows:
        return
    split_dir = cfg.output_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    out_path = split_dir / f"chunk_{chunk_id:06d}.npz"
    save_fn = np.savez_compressed if cfg.save_compressed else np.savez
    save_fn(
        out_path,
        token_sequences=np.array([r["tokens"] for r in rows], dtype=object),
        piano_rolls=np.stack([r["pianoroll"] for r in rows], axis=0).astype(np.float32),
        relative_paths=np.array([r["relative_path"] for r in rows], dtype=object),
        source_types=np.array([r["source_type"] for r in rows], dtype=object),
    )
    manifests[split_name].append(str(out_path.relative_to(cfg.output_dir)))


def process_streaming(cfg: PreprocessConfig, input_index: Dict[str, List[Path]]) -> dict:
    random.seed(cfg.random_seed)
    sources: List[Tuple[str, Path]] = []
    sources.extend(("raw_midi", p) for p in input_index.get("raw_midi", []))
    sources.extend(("lmd_h5", p) for p in input_index.get("lmd_h5", []))

    stats = {
        "seen": 0,
        "parsed_ok": 0,
        "parse_failed": 0,
        "skipped_too_few_notes": 0,
        "raw_midi_used": 0,
        "lmd_h5_used": 0,
        "split_counts": {"train": 0, "val": 0, "test": 0},
    }
    buffers: Dict[str, List[dict]] = {"train": [], "val": [], "test": []}
    chunk_ids = {"train": 0, "val": 0, "test": 0}
    manifests: Dict[str, List[str]] = {"train": [], "val": [], "test": []}

    source_iter: Iterable[Tuple[str, Path]] = sources
    if cfg.show_progress:
        source_iter = tqdm(sources, desc="Preprocessing", unit="file")

    for source_type, path in source_iter:
        stats["seen"] += 1
        notes = parse_raw_midi_file(path) if source_type == "raw_midi" else parse_lmd_h5_file(path)
        if notes is None:
            stats["parse_failed"] += 1
            continue
        if len(notes) < cfg.min_notes_per_track:
            stats["skipped_too_few_notes"] += 1
            continue

        row = {
            "source_type": source_type,
            "relative_path": str(path.relative_to(cfg.project_root)),
            "tokens": notes_to_tokens(notes, cfg.duration_bins, cfg.velocity_bins),
            "pianoroll": notes_to_pianoroll(notes, cfg.fs, cfg.max_roll_steps),
        }
        split_name = split_name_for_example(row["relative_path"], cfg)
        buffers[split_name].append(row)
        stats["split_counts"][split_name] += 1
        stats["parsed_ok"] += 1
        stats["raw_midi_used"] += int(source_type == "raw_midi")
        stats["lmd_h5_used"] += int(source_type == "lmd_h5")

        if len(buffers[split_name]) >= cfg.chunk_size:
            flush_chunk(cfg, split_name, chunk_ids[split_name], buffers[split_name], manifests)
            print(f"Saved chunk {chunk_ids[split_name]} for {split_name} ({len(buffers[split_name])} rows)")
            chunk_ids[split_name] += 1
            buffers[split_name] = []

        if stats["seen"] % cfg.checkpoint_every == 0:
            with open(cfg.output_dir / "preprocessing_checkpoint.json", "w", encoding="utf-8") as f:
                json.dump({"stats": stats, "chunk_counts": {k: len(v) for k, v in manifests.items()}}, f, indent=2)

    for split_name in ("train", "val", "test"):
        if buffers[split_name]:
            flush_chunk(cfg, split_name, chunk_ids[split_name], buffers[split_name], manifests)
            print(f"Saved final chunk {chunk_ids[split_name]} for {split_name} ({len(buffers[split_name])} rows)")

    metadata = {
        "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()},
        "stats": stats,
        "chunk_manifests": manifests,
    }
    with open(cfg.output_dir / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    return metadata


def merge_split_chunks(cfg: PreprocessConfig, split_name: str, manifest: List[str]) -> Optional[Path]:
    if not manifest:
        return None
    all_tokens: List[list] = []
    all_paths: List[str] = []
    all_sources: List[str] = []
    all_rolls: List[np.ndarray] = []
    chunk_iter: Iterable[str] = manifest
    if cfg.show_progress:
        chunk_iter = tqdm(manifest, desc=f"Merging {split_name}", unit="chunk")
    for rel in chunk_iter:
        with np.load(cfg.output_dir / rel, allow_pickle=True) as data:
            all_tokens.extend(data["token_sequences"].tolist())
            all_paths.extend(data["relative_paths"].tolist())
            all_sources.extend(data["source_types"].tolist())
            all_rolls.append(data["piano_rolls"].astype(np.float32))
    out_path = cfg.output_dir / f"{split_name}_dataset.npz"
    save_fn = np.savez_compressed if cfg.save_compressed else np.savez
    save_fn(
        out_path,
        token_sequences=np.array(all_tokens, dtype=object),
        piano_rolls=np.concatenate(all_rolls, axis=0) if all_rolls else np.zeros((0, cfg.max_roll_steps, 128), dtype=np.float32),
        relative_paths=np.array(all_paths, dtype=object),
        source_types=np.array(all_sources, dtype=object),
    )
    return out_path


def validate_architecture_and_outputs(cfg: PreprocessConfig) -> None:
    expected_notebooks = [
        cfg.project_root / "notebooks" / "0_preprocessing.ipynb",
        cfg.project_root / "notebooks" / "task1_lstm_ae.ipynb",
        cfg.project_root / "notebooks" / "task2_vae.ipynb",
        cfg.project_root / "notebooks" / "task3_transformer.ipynb",
        cfg.project_root / "notebooks" / "task4_rlhf.ipynb",
    ]
    project_details_pdf = cfg.project_root / "data" / "Project_Details_of_Neural_Network_Spring_2026.pdf"
    print("Architecture checks:")
    print("- Notebook pipeline exists:", all(p.exists() for p in expected_notebooks))
    print("- Input/output dirs exist:", all(p.exists() for p in [cfg.raw_midi_dir, cfg.output_dir]))
    print("- Project details PDF exists:", project_details_pdf.exists())

    metadata_path = cfg.output_dir / "preprocessing_metadata.json"
    if not metadata_path.exists():
        print("No preprocessing metadata found.")
        return
    with open(metadata_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    manifests = meta.get("chunk_manifests", {})
    print("\nPreprocessing stats:", meta.get("stats", {}))
    print("Chunk counts:", {k: len(v) for k, v in manifests.items()})
    print("\nOutput path checks:")
    for split_name in ("train", "val", "test"):
        split_dir = cfg.output_dir / split_name
        chunk_paths = [cfg.output_dir / rel for rel in manifests.get(split_name, [])]
        print(split_name, "split_dir_exists:", split_dir.exists(), "chunks_exist:", all(p.exists() for p in chunk_paths))
    print("\nProject-fit checks:")
    print("- Uses raw MIDI note events for generation features.")
    print("- Produces token + piano-roll features and train/val/test splits.")
    print("- Saves outputs under data/processed for downstream notebooks.")

In [35]:
inputs = find_input_files(config)
print("Input counts:", {k: len(v) for k, v in inputs.items()})

if config.use_lmd_h5:
    inspect_first_h5_schema(inputs)

Input counts: {'raw_midi': 116189, 'lmd_h5': 0}


In [37]:
metadata = process_streaming(config, inputs)
print("parsed_ok:", metadata["stats"]["parsed_ok"])
print("split_counts:", metadata["stats"]["split_counts"])

Preprocessing:   0%|          | 0/116189 [00:00<?, ?file/s]

Saved chunk 0 for train (256 rows)
Saved chunk 1 for train (256 rows)
Saved chunk 2 for train (256 rows)
Saved chunk 3 for train (256 rows)
Saved chunk 4 for train (256 rows)
Saved chunk 5 for train (256 rows)
Saved chunk 6 for train (256 rows)
Saved chunk 0 for test (256 rows)
Saved chunk 7 for train (256 rows)
Saved chunk 8 for train (256 rows)
Saved chunk 0 for val (256 rows)
Saved chunk 9 for train (256 rows)
Saved chunk 10 for train (256 rows)
Saved chunk 11 for train (256 rows)
Saved chunk 12 for train (256 rows)
Saved chunk 13 for train (256 rows)
Saved chunk 14 for train (256 rows)
Saved chunk 1 for test (256 rows)
Saved chunk 15 for train (256 rows)
Saved chunk 16 for train (256 rows)
Saved chunk 17 for train (256 rows)
Saved chunk 1 for val (256 rows)
Saved chunk 18 for train (256 rows)
Saved chunk 19 for train (256 rows)
Saved chunk 20 for train (256 rows)
Saved chunk 21 for train (256 rows)
Saved chunk 22 for train (256 rows)
Saved chunk 23 for train (256 rows)
Saved chunk 

In [38]:
if config.merge_final_npz:
    with open(config.output_dir / "preprocessing_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    for split_name in ("train", "val", "test"):
        merged = merge_split_chunks(config, split_name, meta["chunk_manifests"][split_name])
        print(split_name, "merged:", merged if merged else "no chunks")
else:
    print("Skipping final merge (merge_final_npz=False).")
    print("Chunked outputs are ready under data/processed/{train,val,test}/")

Skipping final merge (merge_final_npz=False).
Chunked outputs are ready under data/processed/{train,val,test}/


In [39]:
validate_architecture_and_outputs(config)

Architecture checks:
- Notebook pipeline exists: True
- Input/output dirs exist: True
- Project details PDF exists: True

Preprocessing stats: {'seen': 116189, 'parsed_ok': 115176, 'parse_failed': 1012, 'skipped_too_few_notes': 1, 'raw_midi_used': 115176, 'lmd_h5_used': 0, 'split_counts': {'train': 93251, 'val': 10332, 'test': 11593}}
Chunk counts: {'train': 365, 'val': 41, 'test': 46}

Output path checks:
train split_dir_exists: True chunks_exist: True
val split_dir_exists: True chunks_exist: True
test split_dir_exists: True chunks_exist: True

Project-fit checks:
- Uses raw MIDI note events for generation features.
- Produces token + piano-roll features and train/val/test splits.
- Saves outputs under data/processed for downstream notebooks.
